# Datengenerierung und Zugriff
Hier möchte ich Funktionen für verschiedene Parameter kombinieren. Außerdem möchte ich die Generierung der Testdaten von der Abfrage trennen damit die Generierung später einfach durch eine Datenbankabfrage ersetzt werden kann und die Abfrage kürzer und übersichtlicher wird.

In [1]:
import pandas as pd
import numpy as np
from typing import Literal

from generate_data import create_testdata
from access_data import (
    get_meter_metadata,
    get_direction_for_meter_id,
    get_data
)

## Generate

In [4]:
df = create_testdata()
# print(df.to_string())
df

,meter_id,location,timestamp,consumption,generation,forecast
0,mp_1,Wohnhaus,2026-04-12 00:00:00+02:00,1.030472,NaN,False
1,mp_1,Wohnhaus,2026-04-12 00:15:00+02:00,0.928892,NaN,False
2,mp_1,Wohnhaus,2026-04-12 00:30:00+02:00,1.140790,NaN,False
3,mp_1,Wohnhaus,2026-04-12 00:45:00+02:00,1.192585,NaN,False
4,mp_1,Wohnhaus,2026-04-12 01:00:00+02:00,0.936102,NaN,False
...,...,...,...,...,...,...
67,mp_4,Firma,2026-04-16 19:00:00+02:00,NaN,0.607129,True
68,mp_4,Firma,2026-04-16 20:00:00+02:00,NaN,0.823419,True
69,mp_4,Firma,2026-04-16 21:00:00+02:00,NaN,0.804801,True
70,mp_4,Firma,2026-04-16 22:00:00+02:00,NaN,0.784051,True


## Access

### Metadaten

In [3]:
payload = {
    "metadata": get_meter_metadata(df).to_dict(orient="records"),
    "unit": "kWh"
}

payload

{'metadata': [{'meter_id': 'mp_1',
   'location': 'Wohnhaus',
   'direction': 'consumption'},
  {'meter_id': 'mp_2', 'location': 'Wohnhaus', 'direction': 'generation'},
  {'meter_id': 'mp_3', 'location': 'Firma', 'direction': 'consumption'},
  {'meter_id': 'mp_4', 'location': 'Firma', 'direction': 'generation'}],
 'unit': 'kWh'}

### Timeseries

In [4]:
get_data(
    df=df,
    meter_id="mp_1",
    location=None,
    start=None,
    end=None,
    forecast=None
)

,meter_id,location,timestamp,consumption,generation,forecast
0,mp_1,Wohnhaus,2026-04-11 00:00:00+02:00,1.030472,NaN,False
1,mp_1,Wohnhaus,2026-04-11 00:15:00+02:00,0.928892,NaN,False
2,mp_1,Wohnhaus,2026-04-11 00:30:00+02:00,1.140790,NaN,False
3,mp_1,Wohnhaus,2026-04-11 00:45:00+02:00,1.192585,NaN,False
4,mp_1,Wohnhaus,2026-04-11 01:00:00+02:00,0.936102,NaN,False
...,...,...,...,...,...,...
259,mp_1,Wohnhaus,2026-04-15 19:00:00+02:00,0.607129,NaN,True
260,mp_1,Wohnhaus,2026-04-15 20:00:00+02:00,0.823419,NaN,True
261,mp_1,Wohnhaus,2026-04-15 21:00:00+02:00,0.804801,NaN,True
262,mp_1,Wohnhaus,2026-04-15 22:00:00+02:00,0.784051,NaN,True


In [6]:
# optimieren: Redundanzen vermeiden
def get_data_sample(df, meter_id, start, end, max_points=100):
    data = get_data(df, meter_id=meter_id, start=start, end=end)

    # falls zu viele Daten → runter sampeln
    print(len(data))
    if len(data) > max_points:
        data = data.iloc[::len(data)//max_points]

    return data.to_dict(orient="records")

get_data_sample(df, "mp_1", "2026-04-11", "2026-04-16")

264


[{'meter_id': 'mp_1',
  'location': 'Wohnhaus',
  'timestamp': Timestamp('2026-04-11 00:00:00+0200', tz='Europe/Vienna'),
  'consumption': 1.030471707975443,
  'generation': nan,
  'forecast': False},
 {'meter_id': 'mp_1',
  'location': 'Wohnhaus',
  'timestamp': Timestamp('2026-04-11 00:30:00+0200', tz='Europe/Vienna'),
  'consumption': 1.140790180638568,
  'generation': nan,
  'forecast': False},
 {'meter_id': 'mp_1',
  'location': 'Wohnhaus',
  'timestamp': Timestamp('2026-04-11 01:00:00+0200', tz='Europe/Vienna'),
  'consumption': 0.9361021181919439,
  'generation': nan,
  'forecast': False},
 {'meter_id': 'mp_1',
  'location': 'Wohnhaus',
  'timestamp': Timestamp('2026-04-11 01:30:00+0200', tz='Europe/Vienna'),
  'consumption': 1.2088825142500133,
  'generation': nan,
  'forecast': False},
 {'meter_id': 'mp_1',
  'location': 'Wohnhaus',
  'timestamp': Timestamp('2026-04-11 02:00:00+0200', tz='Europe/Vienna'),
  'consumption': 1.2584626585309833,
  'generation': nan,
  'forecast': 

## Plot

In [22]:
def get_direction_for_meter_id(df: pd.DataFrame, meter_id: str) -> str:
    meta = get_meter_metadata(df)
    return meta.loc[
        meta["meter_id"] == meter_id, "direction"
    ].iloc[0]

get_direction_for_meter_id(df, "mp_1")

'consumption'

In [14]:
import matplotlib.pyplot as plt
import uuid
import os

PLOT_DIR = "plots"
os.makedirs(PLOT_DIR, exist_ok=True)

def create_timeseries_plot(df, meter_id=None, start=None, end=None, forecast=None, metric=None):
    # Daten holen
    data = get_data(
        df,
        meter_id=meter_id,
        start=start,
        end=end,
        forecast=forecast
    )

    # richtige Spalte wählen
    if metric not in ["consumption", "generation"]:
        raise ValueError("Invalid metric")

    series = data[["timestamp", metric]].dropna()

    # Plot
    plt.figure()
    plt.plot(series["timestamp"], series[metric])
    plt.xlabel("Time")
    plt.ylabel(f"{metric} (kWh)")
    plt.title(f"{metric} for {meter_id}")

    # Datei speichern
    filename = f"{uuid.uuid4()}.png"
    # filename = f"{meter_id}_{metric}_{start}_{end}.png"
    filepath = os.path.join(PLOT_DIR, filename)

    plt.savefig(filepath)
    plt.close()

    return filename

create_timeseries_plot(
    df=df,
    meter_id="mp_1",
    end="2026-04-16",
    metric="generation")

'ef5959a0-b867-4f7d-9707-3761046b1c32.png'

## Json

In [ ]:
tools = [
  {
    "type": "function",
    "function": {
      "name": "generate_series",
      "description": "Generate synthetic electricity demand (consumption) time series data.",
      "parameters": {
        "type": "object",
        "properties": {
          "start": {
            "type": "string",
            "format": "date-time",
            "description": "Start timestamp in ISO 8601 format (e.g. 2026-01-01T00:00:00)"
          },
          "end": {
            "type": "string",
            "format": "date-time",
            "description": "End timestamp in ISO 8601 format (e.g. 2026-01-01T23:00:00)"
          },
          "pattern": {
            "type": "string",
            "enum": ["residential", "industrial"],
            "description": "Consumption pattern type. Residential has peaks in the morning and evening, industrial peaks during working hours.",
            "default": "residential"
          }
        },
        "required": ["start", "end"]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "generate_production_series",
      "description": "Generate synthetic electricity supply (production) time series data.",
      "parameters": {
        "type": "object",
        "properties": {
          "start": {
            "type": "string",
            "format": "date-time",
            "description": "Start timestamp in ISO 8601 format (e.g. 2026-01-01T00:00:00)"
          },
          "end": {
            "type": "string",
            "format": "date-time",
            "description": "End timestamp in ISO 8601 format (e.g. 2026-01-01T23:00:00)"
          },
          "pattern": {
            "type": "string",
            "enum": ["residential", "industrial"],
            "description": "Production pattern type. Residential may simulate small-scale generation (e.g. solar), industrial simulates large-scale continuous production.",
            "default": "residential"
          }
        },
        "required": ["start", "end"]
      }
    }
  }
]

In [ ]:
def get_timeseries(
    meter_id: str,
    metric: str,
    start_time: str,
    end_time: str,
    include_forecast: bool = True
):
    pass